# `cmsplot` quickstart: histogram utilities

This notebook runs end-to-end with **no CMS/ROOT data required** — it builds
toy histograms directly to demonstrate the core plotting API: `Hist`,
`getRatioHist`, `getSumHist`/`getDiffHist`, and the CMS-style label/axis
helpers.

For the DQM validation plotters (`TrackingDQMPlotter`,
`SecondaryVertexingDQMPlotter`), which read real CMS DQM ROOT files, see the
template scripts in this same `examples/` directory instead — they can't run
without those files, so they're plain `.py` templates rather than an
executable notebook.


In [1]:
import numpy as np
import matplotlib.pyplot as plt
from cmsplot import Hist, getRatioHist, getSumHist, getDiffHist, setStyle, cmslabel, xlabel, ylabel

setStyle()

## Building a `Hist`

A `Hist` is normally constructed from a ROOT file (see `cmsplot.histograms.Hist`),
but you can just as well build one directly by setting `.edges`, `.values`,
and `.errors` — that's all `.plot()` needs. Here we fake a toy efficiency
measurement for two configurations using a binomial pull.

In [2]:
rng = np.random.default_rng(42)

def toy_efficiency_hist(edges, true_eff, n_sim=2000):
    centers = (edges[1:] + edges[:-1]) / 2
    passing = rng.binomial(n_sim, true_eff(centers))
    eff = passing / n_sim
    err = np.sqrt(eff * (1 - eff) / n_sim)

    h = Hist()
    h.edges = edges
    h.values = eff
    h.errors = err
    h.count_quantity = "efficiency"
    h.bin_quantity = "eta"
    return h

edges = np.linspace(-2.5, 2.5, 11)
h_baseline = toy_efficiency_hist(edges, lambda x: 0.92 - 0.03 * x**2)
h_new      = toy_efficiency_hist(edges, lambda x: 0.95 - 0.015 * x**2)

## Overlaying histograms with the CMS style

`Hist.plot()` auto-detects the right plot style (errorbar vs. labeled bar)
and sets axis labels from `count_quantity`/`bin_quantity` via the
`LABEL_DICT` lookup. `cmslabel()` adds the standard CMS experiment label.

In [3]:
fig, ax = plt.subplots(figsize=(8, 7))
h_baseline.plot(ax=ax, marker="o", color="#e41a1c", label="Baseline")
h_new.plot(ax=ax,      marker="s", color="#377eb8", label="New configuration")

ax.set_ylim(0.8, 1.05)
ax.legend(loc="lower left")
cmslabel(llabel="Simulation (Private Work)", rlabel="Toy efficiency example", ax=ax, loc=4)
plt.show()

/tmp/ipykernel_38163/2714820047.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Ratios and differences

`getRatioHist(num, denom)` and `getDiffHist(h1, h2)` / `getSumHist(h1, h2)`
build a new `Hist` with errors propagated in quadrature — handy for a ratio
panel below a main plot (this is exactly what
`DQMPlotter.plotHistogram(..., ratioType="ratio")` does internally).

In [4]:
ratio = getRatioHist(h_new, h_baseline)

fig, ax = plt.subplots(figsize=(8, 4))
ax.axhline(1, linestyle="dashed", color="gray")
ratio.plot(ax=ax, marker="o", color="#377eb8")
ylabel("Ratio", ax=ax)
xlabel("eta", ax=ax)
plt.show()

/tmp/ipykernel_38163/3749603151.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
diff   = getDiffHist(h_new, h_baseline)
summed = getSumHist(h_new, h_baseline)

print("difference (first 3 bins):", diff.values[:3])
print("sum (first 3 bins):       ", summed.values[:3])

difference (first 3 bins): [0.1    0.0745 0.046 ]
sum (first 3 bins):        [1.65   1.7175 1.817 ]


## Next steps

- `Hist2D` follows the same pattern for 2D ROOT histograms (`TH2`), plotted
  via `pcolormesh`.
- `savefig(path)` saves the current figure and creates the output directory
  if needed — used by every example script here instead of `plt.savefig`.
- See `tracking_dqm_example.py` and `secondary_vertexing_dqm_example.py` in
  this directory for the full DQM-plotter workflow against real ROOT files.
